# Generative Adversarial Network (GAN) for MNIST Digits

This notebook implements a **vanilla GAN** that learns to generate realistic handwritten digit images (specifically zeros) from random noise.

### Architecture

* **Generator** — Takes a 100-dimensional random noise vector and transforms it through Dense layers into a 28×28 fake image
* **Discriminator** — A binary classifier that receives 28×28 images and outputs a probability of the image being real
* **GAN (combined)** — Generator + frozen Discriminator; trained end-to-end so the generator learns to fool the discriminator

### Training Loop (Adversarial)

1. **Phase 1 — Train discriminator:** Show it real images (label=1) and fake images from the generator (label=0)
2. **Phase 2 — Train generator (via GAN):** Generate fake images, but label them as real (label=1) — gradients flow back through the frozen discriminator to update only the generator

### Key Concepts

* **Min-max game** — The discriminator tries to distinguish real from fake; the generator tries to produce images the discriminator can’t tell apart
* **Frozen discriminator** — `discriminator.trainable = False` in the GAN ensures only the generator weights update during Phase 2
* **Codings size = 100** — The latent space dimension; each random vector maps to a unique generated image
* **No GPU required** — With only zeros (\~5,900 images) and a shallow architecture, CPU training is feasible (~5-10 min)

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

In [0]:
# Load MNIST: 60K training + 10K test images of handwritten digits (28x28 grayscale)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train/255
X_train = X_train.reshape(-1, 28, 28, 1) *2. -1.


# Filter to only digit '0' — the GAN will learn to generate zeros
only_zeros = X_train[y_train == 0]
print(f"Training on {len(only_zeros)} images of zeros")

In [0]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Reshape , Dropout, LeakyRelu, BatchNormalization, Conv2D, Conv2DTranspose

np.random.seed(42)
tf.random.set_seed(42)
codings_size = 100  # latent space dimension

# --- Discriminator ---
# Binary classifier: takes a 28x28 image and outputs P(real)
discriminator = Sequential()
discriminator.add(Conv2D(64, kernel_size=5, strides=2, padding="same", input_shape=[28,28,1],activation=LeakyRelu(0.3)))
discriminator.add(Dropout(0.5))
discriminator.add(Conv2D(128, kernel_size=5, strides=2, padding="same", activation=LeakyRelu(0.3)))
discriminator.add(Dropout(0.5))
discriminator.add(Flatten())   # 28x28 -> 784
discriminator.add(Dense(1, activation='sigmoid'))   # output: P(image is real)
discriminator.compile(loss='binary_crossentropy', optimizer='adam')

# --- Generator ---
# Maps a random noise vector (100D) to a fake 28x28 image


generator = Sequential()
generator.add(Dense(7*7*128, input_shape=[codings_size]))
generator.add(Reshape([7, 7, 128]))
generator.add(BatchNormalization())
generator.add(Conv2DTranspose(64, kernel_size=5, strides=2, padding="same", activation="relu"))
generator.add(BatchNormalization())
generator.add(Conv2DTranspose(1, kernel_size=5, strides=2 , padding="same", activation="tanh"))
# reshape to image
# Generator is NOT compiled alone — it's trained through the GAN

# --- GAN (Generator + Discriminator) ---
# When training the GAN, only the generator weights update;
# the discriminator is frozen so gradients teach the generator to fool it
discriminator.trainable = False
gan = Sequential([generator, discriminator])
gan.compile(loss='binary_crossentropy', optimizer='adam')

print("Discriminator:")
discriminator.summary()
print("\nGenerator:")
generator.summary()

In [0]:
batch_size = 32
my_data = only_zeros

dataset = tf.data.Dataset.from_tensor_slices(my_data).shuffle(buffer_size=1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)
epochs =1 

generator , discriminator = gan.layers
# Train the GAN
for epoch in range(epochs):
    for X_batch in dataset:

        # --- Train Discriminator ---
        # Generate fake images
        noise = tf.random.normal(shape=[batch_size, codings_size])
        generated_images = generator(noise)
        # Train on real + fake images
        X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
        y1 = tf.constant([[0.0]] * batch_size + [[1.0]] * batch_size)
        discriminator.trainable = True
        discriminator.train_on_batch(X_fake_and_real, y1)

        # --- Train Generator ---
        # Generate noise vectors
        noise = tf.random.normal(shape=[batch_size, codings_size])
        # Train on noise to generate real-looking images
        y2 = tf.constant([[1.0]] * batch_size)
        discriminator.trainable = False
        gan.train_on_batch(noise, y2)
# Generate some fake zeros
noise = tf.random.normal(shape=[32, codings_size])
generated_images = generator(noise)
# Plot them
plt.figure(figsize=(7, 7))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(generated_images[i], interpolation="nearest", cmap="gray")
    plt.axis('off')
plt.show()


